# Lab 4b exercise: Transfer Learning and Fine-Tuning with PyTorch

**Duration:** 1.5 hours  
**Dataset:** CIFAR-10 (3 classes)  
**Model:** Pre-trained ResNet-18

### Tensorflow to pytorch
We will use pytorch instead of tensorflow so that you can use GPU easily if Cuda 11.2-12.6 is installed

### Learning objectives
- Use a pre-trained CNN for a new classification task.
- Freeze pre-trained layers.
- Replace the final classifier.
- Perform transfer learning and fine-tuning.
- Compare the two approaches.

In [1]:
import torch
import torch_directml #using directml because my system has integrated radeon graphics. so switching to that instead of running on cpu. 
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset

device = torch_directml.device()
print('Using:', device)

Using: privateuseone:0


## Task 1: Load CIFAR-10

CIFAR-10 images are 32×32, while the pre-trained ResNet-18 expects larger inputs. We resize them to 224×224 and use ImageNet normalization.

**Question:** Why do we need to resize the images?

In [2]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) #gaussian is optional
])

train_data = datasets.CIFAR10('./data', train=True, download=True, transform=transform)
test_data = datasets.CIFAR10('./data', train=False, download=True, transform=transform)

print('Training:', len(train_data))
print('Testing :', len(test_data))

Files already downloaded and verified
Files already downloaded and verified
Training: 50000
Testing : 10000


## Task 2: Use only three classes

`0 = airplane`, `1 = automobile`, `2 = bird`.

Keeping the dataset small makes the lab faster.

In [3]:
classes_to_use = [0, 1, 2]

train_indices = [i for i, (_, label) in enumerate(train_data) if label in classes_to_use][:6000]
test_indices = [i for i, (_, label) in enumerate(test_data) if label in classes_to_use][:1500]

train_subset = Subset(train_data, train_indices)
test_subset = Subset(test_data, test_indices)

train_loader = DataLoader(train_subset, batch_size=64, shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_subset, batch_size=64, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())

print('Training images:', len(train_subset))
print('Testing images :', len(test_subset))

Training images: 6000
Testing images : 1500


## Task 3: Load a pre-trained ResNet-18

The ImageNet classifier predicts 1000 classes. Our task has only 3 classes, so we replace the final classifier.

**Question:** What information do you expect the pre-trained layers to provide?

In [4]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT).to(device)
print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

## Task 4: Transfer Learning

Freeze all pre-trained layers and train only the new final classifier.

```text
Image → Pre-trained ResNet → Features (FROZEN) → New classifier (TRAINED) → 3 classes
```

In [5]:
for param in model.parameters():
    param.requires_grad = False

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 3).to(device) #3 - output neurons

print('Trainable parameters:')
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)

Trainable parameters:
fc.weight
fc.bias


In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

In [7]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / len(loader), correct / total

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            outputs = model(images)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return correct / total

### Train the transfer-learning model

Train for 3 epochs.

In [8]:
for epoch in range(3):
    loss, accuracy = train_one_epoch(model, train_loader, optimizer, criterion)
    print(f'Epoch {epoch+1}: Loss={loss:.4f}, Accuracy={accuracy:.4f}')

transfer_accuracy = evaluate(model, test_loader)
print('Transfer Learning Test Accuracy:', f'{transfer_accuracy:.4f}')

C:\Users\silen\tfdml_env\lib\site-packages\torch\optim\adam.py:534: UserWarning: The operator 'aten::lerp.Scalar_out' is not currently supported on the DML backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at C:\__w\1\s\pytorch-directml-plugin\torch_directml\csrc\dml\dml_cpu_fallback.cpp:17.)
  torch._foreach_lerp_(device_exp_avgs, device_grads, 1 - beta1)


Epoch 1: Loss=0.5048, Accuracy=0.8275
Epoch 2: Loss=0.2633, Accuracy=0.9153
Epoch 3: Loss=0.2196, Accuracy=0.9283
Transfer Learning Test Accuracy: 0.9287


## Task 5: Fine-Tuning

Unfreeze the last ResNet block (`layer4`). Earlier layers remain frozen.

**Question:** Why should we usually use a smaller learning rate during fine-tuning?

In [9]:
for param in model.layer4.parameters():
    param.requires_grad = True

print('Trainable parameters:')
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)

Trainable parameters:
layer4.0.conv1.weight
layer4.0.bn1.weight
layer4.0.bn1.bias
layer4.0.conv2.weight
layer4.0.bn2.weight
layer4.0.bn2.bias
layer4.0.downsample.0.weight
layer4.0.downsample.1.weight
layer4.0.downsample.1.bias
layer4.1.conv1.weight
layer4.1.bn1.weight
layer4.1.bn1.bias
layer4.1.conv2.weight
layer4.1.bn2.weight
layer4.1.bn2.bias
fc.weight
fc.bias


In [10]:
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0001)

In [11]:
for epoch in range(3):
    loss, accuracy = train_one_epoch(model, train_loader, optimizer, criterion)
    print(f'Fine-tuning Epoch {epoch+1}: Loss={loss:.4f}, Accuracy={accuracy:.4f}')

fine_tuned_accuracy = evaluate(model, test_loader)
print('Fine-Tuned Test Accuracy:', f'{fine_tuned_accuracy:.4f}')

Fine-tuning Epoch 1: Loss=0.1326, Accuracy=0.9530
Fine-tuning Epoch 2: Loss=0.0184, Accuracy=0.9952
Fine-tuning Epoch 3: Loss=0.0085, Accuracy=0.9987
Fine-Tuned Test Accuracy: 0.9640


## Task 6: Compare

| Approach | Trainable part | Learning rate | Test Accuracy |
|---|---|---:|---:|
| Transfer learning | FC only | 0.001 | 0.9287 |
| Fine-tuning | Layer 4 + FC | 0.0001 | 0.9640 |

1. Which approach performed better?

**Answer:** Fine-tuning performed better.

Transfer learning :92.87 | Fine-tuning       :96.40%

2. Why is the learning rate smaller during fine-tuning?

**Answer:** To make small adjustments to the learned weights of 4th layer rather than changing them drastically.

3. Why don't we immediately unfreeze the entire network?

**Answer:** If we unfreeze the entire network than all the weights of pre-trained model get changed while re-training.
 And that too with the small dataset which cause the overfitting as training the big model with small dataset.
 Also requires more computation.

4. Does fine-tuning always improve accuracy?

**Answer:** Yes, with small learning weights and less number of unfreezen layers.

No, when the dataset is completely different and small while training big model.

## Tasks for self study

### Experiment A — Unfreeze more layers
Try unfreezing both `layer3` and `layer4` and compare the result.

### Experiment B — Change the learning rate
Try `0.001`, `0.0001`, and `0.00001`.

| Fine-tuning LR | Test Accuracy |
|---:|---:|
| 0.001 | |
| 0.0001 | |
| 0.00001 | |

**When would you choose transfer learning instead of fine-tuning?**

**Experiment A:**

In [13]:
# With 2 unfreezen layers (4 & 3)

for param in model.layer4.parameters():
    param.requires_grad = True

for param in model.layer3.parameters():
    param.requires_grad = True

print('Trainable parameters:')
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)

Trainable parameters:
layer3.0.conv1.weight
layer3.0.bn1.weight
layer3.0.bn1.bias
layer3.0.conv2.weight
layer3.0.bn2.weight
layer3.0.bn2.bias
layer3.0.downsample.0.weight
layer3.0.downsample.1.weight
layer3.0.downsample.1.bias
layer3.1.conv1.weight
layer3.1.bn1.weight
layer3.1.bn1.bias
layer3.1.conv2.weight
layer3.1.bn2.weight
layer3.1.bn2.bias
layer4.0.conv1.weight
layer4.0.bn1.weight
layer4.0.bn1.bias
layer4.0.conv2.weight
layer4.0.bn2.weight
layer4.0.bn2.bias
layer4.0.downsample.0.weight
layer4.0.downsample.1.weight
layer4.0.downsample.1.bias
layer4.1.conv1.weight
layer4.1.bn1.weight
layer4.1.bn1.bias
layer4.1.conv2.weight
layer4.1.bn2.weight
layer4.1.bn2.bias
fc.weight
fc.bias


In [14]:
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.0001
)

for epoch in range(3):
    loss, accuracy = train_one_epoch(
        model, train_loader, optimizer, criterion
    )
    print(f'Epoch {epoch+1}: Loss={loss:.4f}, Accuracy={accuracy:.4f}')

accuracy_layer3_layer4 = evaluate(model, test_loader)
print('Test Accuracy:', f'{accuracy_layer3_layer4:.4f}')

Epoch 1: Loss=0.0173, Accuracy=0.9947
Epoch 2: Loss=0.0186, Accuracy=0.9937
Epoch 3: Loss=0.0138, Accuracy=0.9958
Test Accuracy: 0.9687


In [15]:
## With 3 unfreezen layers (4, 3 & 2)

for param in model.layer4.parameters():
    param.requires_grad = True

for param in model.layer3.parameters():
    param.requires_grad = True

for param in model.layer2.parameters():
    param.requires_grad = True

print('Trainable parameters:')
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)

Trainable parameters:
layer2.0.conv1.weight
layer2.0.bn1.weight
layer2.0.bn1.bias
layer2.0.conv2.weight
layer2.0.bn2.weight
layer2.0.bn2.bias
layer2.0.downsample.0.weight
layer2.0.downsample.1.weight
layer2.0.downsample.1.bias
layer2.1.conv1.weight
layer2.1.bn1.weight
layer2.1.bn1.bias
layer2.1.conv2.weight
layer2.1.bn2.weight
layer2.1.bn2.bias
layer3.0.conv1.weight
layer3.0.bn1.weight
layer3.0.bn1.bias
layer3.0.conv2.weight
layer3.0.bn2.weight
layer3.0.bn2.bias
layer3.0.downsample.0.weight
layer3.0.downsample.1.weight
layer3.0.downsample.1.bias
layer3.1.conv1.weight
layer3.1.bn1.weight
layer3.1.bn1.bias
layer3.1.conv2.weight
layer3.1.bn2.weight
layer3.1.bn2.bias
layer4.0.conv1.weight
layer4.0.bn1.weight
layer4.0.bn1.bias
layer4.0.conv2.weight
layer4.0.bn2.weight
layer4.0.bn2.bias
layer4.0.downsample.0.weight
layer4.0.downsample.1.weight
layer4.0.downsample.1.bias
layer4.1.conv1.weight
layer4.1.bn1.weight
layer4.1.bn1.bias
layer4.1.conv2.weight
layer4.1.bn2.weight
layer4.1.bn2.bias
fc.

In [16]:
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.0001
)

for epoch in range(3):
    loss, accuracy = train_one_epoch(
        model, train_loader, optimizer, criterion
    )
    print(f'Epoch {epoch+1}: Loss={loss:.4f}, Accuracy={accuracy:.4f}')

accuracy_layer2_layer3_layer4 = evaluate(model, test_loader)
print('Three-Layer Fine-Tuning Test Accuracy:',
      f'{accuracy_layer2_layer3_layer4:.4f}')

Epoch 1: Loss=0.0202, Accuracy=0.9933
Epoch 2: Loss=0.0200, Accuracy=0.9938
Epoch 3: Loss=0.0129, Accuracy=0.9965
Three-Layer Fine-Tuning Test Accuracy: 0.9587


Unfreezing `layer3` in addition to `layer4` improved the test accuracy, from 96.40% (Layer 4 only) up to 96.87%. Unfreezing `layer2` as well made it worse, dropping to 95.87%. This suggests there is a sweet spot: giving the model a little more capacity to adapt (Layer 3 + Layer 4) helped, but unfreezing too much let the earlier, more general features drift and hurt performance, likely due to overfitting on such a small dataset.

**Experiment B:**

In [17]:
for param in model.layer4.parameters():
    param.requires_grad = True

print('Trainable parameters:')
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)

Trainable parameters:
layer2.0.conv1.weight
layer2.0.bn1.weight
layer2.0.bn1.bias
layer2.0.conv2.weight
layer2.0.bn2.weight
layer2.0.bn2.bias
layer2.0.downsample.0.weight
layer2.0.downsample.1.weight
layer2.0.downsample.1.bias
layer2.1.conv1.weight
layer2.1.bn1.weight
layer2.1.bn1.bias
layer2.1.conv2.weight
layer2.1.bn2.weight
layer2.1.bn2.bias
layer3.0.conv1.weight
layer3.0.bn1.weight
layer3.0.bn1.bias
layer3.0.conv2.weight
layer3.0.bn2.weight
layer3.0.bn2.bias
layer3.0.downsample.0.weight
layer3.0.downsample.1.weight
layer3.0.downsample.1.bias
layer3.1.conv1.weight
layer3.1.bn1.weight
layer3.1.bn1.bias
layer3.1.conv2.weight
layer3.1.bn2.weight
layer3.1.bn2.bias
layer4.0.conv1.weight
layer4.0.bn1.weight
layer4.0.bn1.bias
layer4.0.conv2.weight
layer4.0.bn2.weight
layer4.0.bn2.bias
layer4.0.downsample.0.weight
layer4.0.downsample.1.weight
layer4.0.downsample.1.bias
layer4.1.conv1.weight
layer4.1.bn1.weight
layer4.1.bn1.bias
layer4.1.conv2.weight
layer4.1.bn2.weight
layer4.1.bn2.bias
fc.

In [18]:
# lr = 0.001


optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.001
)

for epoch in range(3):
    loss, accuracy = train_one_epoch(
        model, train_loader, optimizer, criterion
    )
    print(f'Epoch {epoch+1}: Loss={loss:.4f}, Accuracy={accuracy:.4f}')

accuracy_lr_001 = evaluate(model, test_loader)

print('Test Accuracy with LR 0.001:',
      f'{accuracy_lr_001:.4f}')

Epoch 1: Loss=0.2458, Accuracy=0.9223
Epoch 2: Loss=0.0931, Accuracy=0.9683
Epoch 3: Loss=0.0846, Accuracy=0.9683
Test Accuracy with LR 0.001: 0.9447


In [19]:
# lr = 0.0001

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.0001
)

for epoch in range(3):
    loss, accuracy = train_one_epoch(
        model, train_loader, optimizer, criterion
    )
    print(f'Epoch {epoch+1}: Loss={loss:.4f}, Accuracy={accuracy:.4f}')

accuracy_lr_0001 = evaluate(model, test_loader)

print('Test Accuracy with LR 0.0001:',
      f'{accuracy_lr_0001:.4f}')

Epoch 1: Loss=0.0241, Accuracy=0.9928
Epoch 2: Loss=0.0073, Accuracy=0.9977
Epoch 3: Loss=0.0024, Accuracy=0.9995
Test Accuracy with LR 0.0001: 0.9707


In [20]:
# lr = 0.00001

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.00001
)

for epoch in range(3):
    loss, accuracy = train_one_epoch(
        model, train_loader, optimizer, criterion
    )
    print(f'Epoch {epoch+1}: Loss={loss:.4f}, Accuracy={accuracy:.4f}')

accuracy_lr_00001 = evaluate(model, test_loader)

print('Test Accuracy with LR 0.00001:',
      f'{accuracy_lr_00001:.4f}')

Epoch 1: Loss=0.0016, Accuracy=0.9998
Epoch 2: Loss=0.0010, Accuracy=0.9998
Epoch 3: Loss=0.0006, Accuracy=1.0000
Test Accuracy with LR 0.00001: 0.9727


**Comparison**

Learning Rate Comparison:

LR = 0.001: Test Accuracy = 0.9447 (94.47%)

LR = 0.0001: Test Accuracy = 0.9707 (97.07%)

LR = 0.00001: Test Accuracy = 0.9727 (97.27%)

Best Learning Rate: 0.00001

Best Test Accuracy: 0.9727

**Question:** When would you choose transfer learning instead of fine-tuning?


**Answer:** 

I would choose transfer learning instead of fine-tuning when I have a small dataset or limited computational resources. In transfer learning, the pre-trained layers are kept frozen and only the final classifier is trained, making the training faster and reducing the risk of overfitting.

Fine-tuning is more useful when I have enough data and want the pre-trained features to adapt more specifically to the new dataset.